# Module 3: Data Cleaning & Transformation

Topics Covered:
- 3.1 Handling Missing Values (isna, notna, fillna, dropna with subsets & thresholds)
- 3.2 Detecting & Removing Duplicates (duplicated, drop_duplicates)
- 3.3 Type Conversions (astype, pd.to_numeric, pd.to_datetime with error coercion)
- 3.4 Vectorized String Operations (.str.strip, .lower, .contains, .replace, regex extract)
- 3.5 Applying Functions (.apply, .map, .transform)
- 3.6 Filtering Rows (boolean masks, compound conditions &, |, ~, and df.query())
- 3.7 Sorting & Ranking (sort_values, multi-column sorting, rank)
- 3.8 Replacing Values & Conditional Masking (.replace, .where, .mask, np.select)
- 3.9 Renaming & Standardizing Dirty Data (clean column headers, strip whitespace)
- 3.10 Outlier Detection & Treatment (IQR fence clipping, Z-score thresholds)
- 3.11 Hands-on Practice Challenge with complete solution!

In [1]:
import numpy as np
import pandas as pd

#### 3.1 Handling Missing Values (NaN / None)

In [2]:
# Real-world dirty dataset: Customer registrations
raw_data = {
    "customer_id": [101, 102, 103, 104, 105, 106],
    "email": ["alice@gmail.com", "BOB@YAHOO.COM", None, "david@outlook.com", "eva@gmail.com", None],
    "age": [25, np.nan, 32, 45, np.nan, 29],
    "income": [65000.0, 72000.0, np.nan, 120000.0, 58000.0, 61000.0],
    "signup_channel": ["Google", "Facebook", "Google", None, "Referral", "Google"]
}

df = pd.DataFrame(raw_data)
print("Raw Dirty Dataset:\n", df)

Raw Dirty Dataset:
    customer_id              email   age    income signup_channel
0          101    alice@gmail.com  25.0   65000.0         Google
1          102      BOB@YAHOO.COM   NaN   72000.0       Facebook
2          103                NaN  32.0       NaN         Google
3          104  david@outlook.com  45.0  120000.0            NaN
4          105      eva@gmail.com   NaN   58000.0       Referral
5          106                NaN  29.0   61000.0         Google


In [5]:
# 1. Detection: Count missing values per column
print("\nMissing values per column:\n", df.isna())
print("\nMissing values per column:\n", df.isna().sum())



Missing values per column:
    customer_id  email    age  income  signup_channel
0        False  False  False   False           False
1        False  False   True   False           False
2        False   True  False    True           False
3        False  False  False   False            True
4        False  False   True   False           False
5        False   True  False   False           False

Missing values per column:
 customer_id       0
email             2
age               2
income            1
signup_channel    1
dtype: int64


In [7]:
# 2. Imputation: Fill missing numeric 'age' with column median
median_age = df["age"].median()
df["age_filled"] = df["age"].fillna(median_age)
print(f"\nImputed age (median = {median_age:.1f}):\n", df[["age", "age_filled"]])

# Forward fill / Backward fill (useful for time series & sensor logs):
df["signup_channel_ffill"] = df["signup_channel"].ffill()
print("\nForward-filled signup channel:\n", df[["signup_channel", "signup_channel_ffill"]])



Imputed age (median = 30.5):
     age  age_filled
0  25.0        25.0
1   NaN        30.5
2  32.0        32.0
3  45.0        45.0
4   NaN        30.5
5  29.0        29.0

Forward-filled signup channel:
   signup_channel signup_channel_ffill
0         Google               Google
1       Facebook             Facebook
2         Google               Google
3            NaN               Google
4       Referral             Referral
5         Google               Google


In [8]:
# 3. Removal: Drop rows where critical primary identifiers (email) are missing
df_clean_email = df.dropna(subset=["email"])
print(f"\nRows remaining after dropping missing emails: {len(df_clean_email)} / {len(df)}")


Rows remaining after dropping missing emails: 4 / 6


#### 3.2 Duplicate Detection and Removal

In [13]:
orders_with_dupes = pd.DataFrame({
    "order_id": ["ORD-1", "ORD-2", "ORD-2", "ORD-3", "ORD-3", "ORD-3"],
    "user_id": [10, 20, 20, 30, 30, 30],
    "amount": [150.0, 200.0, 200.0, 75.0, 75.0, 95.0]  # Note: 3rd ORD-3 has different amount
})
print("Orders with duplicate records:\n", orders_with_dupes)

# Check which rows are duplicates:
print("\nDuplicate mask (is row identical to previous?):\n", orders_with_dupes.duplicated())

# Drop exact duplicate rows:
exact_deduped = orders_with_dupes.drop_duplicates()
print("\nExact deduped rows:\n", exact_deduped)

# Drop duplicates based on specific business key (order_id), keeping the first:
deduped_by_id = orders_with_dupes.drop_duplicates(subset=["order_id"], keep="first")
print("\nDeduped by order_id (keeping first):\n", deduped_by_id)


Orders with duplicate records:
   order_id  user_id  amount
0    ORD-1       10   150.0
1    ORD-2       20   200.0
2    ORD-2       20   200.0
3    ORD-3       30    75.0
4    ORD-3       30    75.0
5    ORD-3       30    95.0

Duplicate mask (is row identical to previous?):
 0    False
1    False
2     True
3    False
4     True
5    False
dtype: bool

Exact deduped rows:
   order_id  user_id  amount
0    ORD-1       10   150.0
1    ORD-2       20   200.0
3    ORD-3       30    75.0
5    ORD-3       30    95.0

Deduped by order_id (keeping first):
   order_id  user_id  amount
0    ORD-1       10   150.0
1    ORD-2       20   200.0
3    ORD-3       30    75.0


#### 3.3 Type Conversions (astype, pd.to_numeric, pd.to_datetime)

In [14]:
dirty_types = pd.DataFrame({
    "price_str": ["$19.99", "$45.50", "Free", "$89.00", "N/A"],
    "date_str": ["2026-01-15", "15/02/2026", "2026-03-30", "invalid_date", "2026-05-12"],
    "is_active_int": [1, 0, 1, 1, 0]
})
print("Dirty Types DataFrame:\n", dirty_types)

Dirty Types DataFrame:
   price_str      date_str  is_active_int
0    $19.99    2026-01-15              1
1    $45.50    15/02/2026              0
2      Free    2026-03-30              1
3    $89.00  invalid_date              1
4       N/A    2026-05-12              0


In [15]:
# 1. Clean string currency and safely parse to float (coerce invalid strings like 'Free' to NaN):
clean_prices = dirty_types["price_str"].str.replace("$", "", regex=False)
dirty_types["price_numeric"] = pd.to_numeric(clean_prices, errors="coerce")
print("\nSafely parsed numeric prices (errors='coerce'):\n", dirty_types[["price_str", "price_numeric"]])



Safely parsed numeric prices (errors='coerce'):
   price_str  price_numeric
0    $19.99          19.99
1    $45.50          45.50
2      Free            NaN
3    $89.00          89.00
4       N/A            NaN


In [16]:
# 2. Parse mixed dates safely (coercing 'invalid_date' to NaT - Not a Time):
dirty_types["parsed_date"] = pd.to_datetime(dirty_types["date_str"], format="mixed", errors="coerce")
print("\nSafely parsed datetimes:\n", dirty_types[["date_str", "parsed_date"]])


Safely parsed datetimes:
        date_str parsed_date
0    2026-01-15  2026-01-15
1    15/02/2026  2026-02-15
2    2026-03-30  2026-03-30
3  invalid_date         NaT
4    2026-05-12  2026-05-12


In [17]:
# 3. Boolean casting:
dirty_types["is_active"] = dirty_types["is_active_int"].astype(bool)
print("\nCast integer 1/0 to bool:\n", dirty_types[["is_active_int", "is_active"]])


Cast integer 1/0 to bool:
    is_active_int  is_active
0              1       True
1              0      False
2              1       True
3              1       True
4              0      False


#### 3.4 Vectorized String Operations (.str Accessor)

In [22]:
contacts = pd.DataFrame({
    "raw_name": ["  Alice Johnson ", "BOB SMITH", "Charlie Brown  ", "  diana Prince "],
    "email": ["alice@gmail.com", "bob@corporate.co.uk", "charlie@gmail.com", "diana@yahoo.com"],
    "phone": ["+1-555-0192", "+1-555-0144", "+44-20-7946", "+1-555-0189"]
})

# Standardize names: strip leading/trailing whitespace, convert to Title Case
contacts["clean_name"] = contacts["raw_name"].str.strip().str.title()

# Filter emails containing 'gmail':
gmail_users = contacts[contacts["email"].str.contains("gmail", case=False, na=False)]
print("Gmail users filtered via .str.contains():\n", gmail_users[["clean_name", "email"]])

Gmail users filtered via .str.contains():
       clean_name              email
0  Alice Johnson    alice@gmail.com
2  Charlie Brown  charlie@gmail.com


In [24]:
# Extract domain using regex capture group:
contacts["email_domain"] = contacts["email"].str.extract(r"@([\w\.-]+)")
print("\nExtracted Email Domains:\n", contacts[["clean_name", "email", "email_domain"]])


Extracted Email Domains:
       clean_name                email     email_domain
0  Alice Johnson      alice@gmail.com        gmail.com
1      Bob Smith  bob@corporate.co.uk  corporate.co.uk
2  Charlie Brown    charlie@gmail.com        gmail.com
3   Diana Prince      diana@yahoo.com        yahoo.com


#### 3.5 Applying Functions (.apply, .map, .transform)

In [26]:
scores_df = pd.DataFrame({
    "student": ["Alex", "Beth", "Chris", "Dave"],
    "score": [88, 92, 54, 76],
    "department": ["CS", "CS", "Math", "Math"]
})

# .map() with a dictionary: Map department code to full department name
dept_names = {"CS": "Computer Science", "Math": "Mathematics"}
scores_df["dept_full"] = scores_df["department"].map(dept_names)

print("scores_df :\n", scores_df)

scores_df :
   student  score department         dept_full
0    Alex     88         CS  Computer Science
1    Beth     92         CS  Computer Science
2   Chris     54       Math       Mathematics
3    Dave     76       Math       Mathematics


In [27]:
# .apply() with custom business logic (e.g. grading system):
def grade_classifier(score: float) -> str:
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    return "F"

scores_df["grade"] = scores_df["score"].apply(grade_classifier)
print("Scores with mapped department and applied grades:\n", scores_df)

Scores with mapped department and applied grades:
   student  score department         dept_full grade
0    Alex     88         CS  Computer Science     B
1    Beth     92         CS  Computer Science     A
2   Chris     54       Math       Mathematics     F
3    Dave     76       Math       Mathematics     C


In [28]:
# .transform() for group-level operations (e.g., score deviation from department mean):
scores_df["dept_mean"] = scores_df.groupby("department")["score"].transform("mean")
scores_df["score_diff_from_dept"] = scores_df["score"] - scores_df["dept_mean"]
print("\nDepartment relative performance with .transform():\n",
      scores_df[["student", "score", "dept_full", "dept_mean", "score_diff_from_dept"]])


Department relative performance with .transform():
   student  score         dept_full  dept_mean  score_diff_from_dept
0    Alex     88  Computer Science       90.0                  -2.0
1    Beth     92  Computer Science       90.0                   2.0
2   Chris     54       Mathematics       65.0                 -11.0
3    Dave     76       Mathematics       65.0                  11.0


#### 3.6 Filtering with Compound Conditions and df.query()

In [29]:
inventory = pd.DataFrame({
    "item": ["Laptop", "Monitor", "Keyboard", "Mouse", "Desk", "Chair"],
    "category": ["Electronics", "Electronics", "Electronics", "Electronics", "Furniture", "Furniture"],
    "price": [1200.0, 300.0, 85.0, 30.0, 450.0, 220.0],
    "in_stock": [True, True, False, True, True, False]
})

print("inventory :\n ", inventory)

inventory :
         item     category   price  in_stock
0    Laptop  Electronics  1200.0      True
1   Monitor  Electronics   300.0      True
2  Keyboard  Electronics    85.0     False
3     Mouse  Electronics    30.0      True
4      Desk    Furniture   450.0      True
5     Chair    Furniture   220.0     False


In [30]:
# Method 1: Boolean mask with bitwise & (AND), | (OR), ~ (NOT)
# Note: Always wrap each condition in parentheses!
mask = (inventory["category"] == "Electronics") & (inventory["price"] > 100.0) & inventory["in_stock"]
print("Filtered via Boolean Mask:\n", inventory[mask])

Filtered via Boolean Mask:
       item     category   price  in_stock
0   Laptop  Electronics  1200.0      True
1  Monitor  Electronics   300.0      True


In [31]:
# Method 2: Expressive SQL-like df.query() (Cleaner syntax for backend developers!)
query_result = inventory.query("category == 'Electronics' and price > 100.0 and in_stock == True")
print("\nFiltered via df.query():\n", query_result)


Filtered via df.query():
       item     category   price  in_stock
0   Laptop  Electronics  1200.0      True
1  Monitor  Electronics   300.0      True


#### 3.7 Sorting & Ranking

In [32]:
# Sort by multiple columns: category ascending, price descending
sorted_inventory = inventory.sort_values(by=["category", "price"], ascending=[True, False])
print("Sorted by Category ASC, Price DESC:\n", sorted_inventory)

Sorted by Category ASC, Price DESC:
        item     category   price  in_stock
0    Laptop  Electronics  1200.0      True
1   Monitor  Electronics   300.0      True
2  Keyboard  Electronics    85.0     False
3     Mouse  Electronics    30.0      True
4      Desk    Furniture   450.0      True
5     Chair    Furniture   220.0     False


In [33]:
# Rank items by price within each category:
inventory["price_rank_in_cat"] = inventory.groupby("category")["price"].rank(ascending=False, method="dense")
print("\nRanked within category by price:\n", inventory[["category", "item", "price", "price_rank_in_cat"]])


Ranked within category by price:
       category      item   price  price_rank_in_cat
0  Electronics    Laptop  1200.0                1.0
1  Electronics   Monitor   300.0                2.0
2  Electronics  Keyboard    85.0                3.0
3  Electronics     Mouse    30.0                4.0
4    Furniture      Desk   450.0                1.0
5    Furniture     Chair   220.0                2.0


#### 3.8 Conditional Replacement: .where, .mask, and np.select

In [34]:
sales_reps = pd.DataFrame({
    "rep": ["John", "Sara", "Mike", "Anna"],
    "deals_closed": [12, 4, 18, 8],
    "revenue": [120000, 35000, 210000, 80000]
})

# np.select() is the Python equivalent of SQL CASE WHEN ... THEN ... ELSE
conditions = [
    sales_reps["revenue"] >= 150000,
    sales_reps["revenue"] >= 75000,
    sales_reps["revenue"] < 75000
]
choices = ["Tier 1 (Elite)", "Tier 2 (Core)", "Tier 3 (Developing)"]
sales_reps["performance_tier"] = np.select(conditions, choices, default="Unranked")
print("SQL CASE WHEN equivalent via np.select():\n", sales_reps)

SQL CASE WHEN equivalent via np.select():
     rep  deals_closed  revenue     performance_tier
0  John            12   120000        Tier 2 (Core)
1  Sara             4    35000  Tier 3 (Developing)
2  Mike            18   210000       Tier 1 (Elite)
3  Anna             8    80000        Tier 2 (Core)


#### 3.9 Outlier Detection & Treatment (IQR Rule & Clipping)

In [35]:
# Salary dataset with extreme outliers (e.g. CEO compensation or data entry errors)
salaries = pd.Series([45000, 48000, 52000, 50000, 55000, 53000, 49000, 2500000, 15000])  # 2.5M is an outlier!

# Compute Interquartile Range (IQR)
q1 = salaries.quantile(0.25)
q3 = salaries.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Valid Non-Outlier Range: [{lower_fence:.1f}, {upper_fence:.1f}]")

Q1: 48000.0, Q3: 53000.0, IQR: 5000.0
Valid Non-Outlier Range: [40500.0, 60500.0]


In [36]:
# Outliers identified:
outliers = salaries[(salaries < lower_fence) | (salaries > upper_fence)]
print("Detected Outliers:\n", outliers)

# Treat outliers by capping/clipping to the upper and lower fences:
salaries_clipped = salaries.clip(lower=lower_fence, upper=upper_fence)
print("\nSalaries after .clip(lower_fence, upper_fence):\n", salaries_clipped)

Detected Outliers:
 7    2500000
8      15000
dtype: int64

Salaries after .clip(lower_fence, upper_fence):
 0    45000
1    48000
2    52000
3    50000
4    55000
5    53000
6    49000
7    60500
8    40500
dtype: int64


#### 3.10 Hands-on Practice Challenge

In [ ]:
print("""
PRACTICE EXERCISE:
Given a messy e-commerce customer transaction log:
df_messy = pd.DataFrame({
    "trans_id": [1, 2, 2, 3, 4, 5],
    "user_email": ["john@work.com", "SARA@OUTLOOK.COM", "sara@outlook.com", None, "mike@work.com", "invalid_email"],
    "amount": ["$120.50", "$45.00", "$45.00", "$999.00", "Refunded", "$35.20"],
    "discount": [10.0, np.nan, np.nan, 20.0, 0.0, np.nan]
})

Your Task:
1. Drop duplicate rows based on 'trans_id', keeping the first occurrence.
2. Remove rows where 'user_email' is missing (None/NaN) or does not contain '@'.
3. Clean 'amount': strip '$', coerce invalid values ('Refunded') to NaN, and fill NaN with 0.0.
4. Fill missing 'discount' values with 0.0.
5. Add a boolean column 'is_corporate_user' that is True if user_email ends with '@work.com'.
""")

In [38]:
# --- Challenge Solution ---
df_messy = pd.DataFrame({
    "trans_id": [1, 2, 2, 3, 4, 5],
    "user_email": ["john@work.com", "SARA@OUTLOOK.COM", "sara@outlook.com", None, "mike@work.com", "invalid_email"],
    "amount": ["$120.50", "$45.00", "$45.00", "$999.00", "Refunded", "$35.20"],
    "discount": [10.0, np.nan, np.nan, 20.0, 0.0, np.nan]
})

In [39]:
# 1. Dedup
df_clean = df_messy.drop_duplicates(subset=["trans_id"], keep="first").copy()